# 1. Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings
import numpy as np
from IPython.display import clear_output
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# 2. Defining the Product Information and Location

In [2]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][9]
print(search_text)
Source="Go-Parts"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'
filename=Source+'ProductLinks_'+search_text+'.xlsx'
df1=pd.read_excel(IFolder+'\\'+filename)
df1

                                  Product Name
0                                Ignition Coil
1                      Windshield Washer Pumps
2                        coupler trailer locks
3          Adjustable Trailer Hitch Ball Mount
4                               Vacuum Cleaner
5                                   Spark Plug
6              bluetooth Enabled Trailer locks
7                          Power Steering Hose
8   Power Steering Pressure Line Hose Assembly
9                       Washer Fluid Reservoir
10          Hitch Ball Mount with Weight Scale
11                               LED Headlamps
12                             LED flashlights
13                   Fiberglass Tonneau Covers
14                    Aluminium Tonneau Covers
15                     Hardfold Tonneau Covers
16                            Spark Plug Wires
17                 washer fluid reservoir tank
18                   Non Automotive Gas Struts
Washer Fluid Reservoir


,Links,Name
0,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ AC1288104)
1,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir - Front (Dorman 603-211)
2,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ CH1288271)
3,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ GM1288271)
4,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ HY1288112)
...,...,...
282,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ SU1288108)
283,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ VW1288135)
284,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ AU1288106)
285,https://www.go-parts.com/ps-product-new.php?sk...,Washer Fluid Reservoir (LKQ HO1288213)


In [3]:
df1.loc[1,"Name"]

'Washer Fluid Reservoir - Front (Dorman 603-211)'

In [4]:
links=[]
for i in range(len(df1)):
    if " " in df1.loc[i,"Name"]:
        links.append(df1['Links'][i])

links=set(links)
links=list(links)

In [5]:
length=len(links)
print(length)

287


# 3. Setting Webdriver and Website Specific Information

In [6]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 5)
driver.get('https://www.go-parts.com/')
pretext="https://www.go-parts.com/ps-product-new.php?sku="

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [32]:
cols =['Sl.No','Attributes','Details'] 
df = pd.DataFrame(columns=cols)
df
count=0

In [10]:
driver.get(links[12])

In [33]:
from tqdm import tqdm
for i in tqdm(range(length)):
    driver.get(links[i])
    df.loc[count,'Sl.No']=int(i)
    sleep(3)
    try:
        df.loc[count,'Name']=driver.find_element(By.CSS_SELECTOR,'[class="product-name"]').text
        df.loc[count,'Links']=pretext+driver.find_element(By.CSS_SELECTOR,'[class="sku-style"]').text.rsplit(" ",1)[1]
    except:
        pass
    try:
        df.loc[count,'Part Number']=driver.find_element(By.CSS_SELECTOR,'[class="product-name"]').text.split("(")[1].replace(")","").rsplit(" ",1)[1]
        df.loc[count,'Brand']=driver.find_element(By.CSS_SELECTOR,'[class="product-name"]').text.split("(")[1].replace(")","").rsplit(" ",1)[0]
    except:
        pass
    try:
        df.loc[count,'Current Price']=float(driver.find_element(By.CSS_SELECTOR,'[class="regular-price"]').text.replace("\n","").replace("$","").replace(",",""))
    except:
        pass
    try:
        l=driver.find_element(By.CSS_SELECTOR,'[class="std"]').text.split('\n')
        Alist=[]
        Dlist=[]
        for i in range(len(l)):
            if len(l[i])<200 and len(l[i])>1:
                if ':' in l[i]:
                    if not l[i].endswith(":"):
                        Alist.append(l[i])        
        df.at[count,"Attributes"]=list(map(lambda x: x.replace(": ",":"),Alist))
        Dlist=[ele for ele in l if ele not in Alist]
        df.at[count,"Details"]=Dlist
    except:
        pass
    count=count+1    
    balanceitem=length-i
    #timeremaining(balanceitem,i)
    #clear_output(wait=True)

































































































































































































































































































































































































































































































































































































100%|██████████| 287/287 [25:06<00:00,  5.25s/it]


In [34]:
print(df.shape)
df

(287, 8)


,Sl.No,Attributes,Details,Name,Links,Part Number,Brand,Current Price
0,0,"[Mounting Bracket Included:Yes, Mounting Hardw...",[Dual Coolant / Windshield Washer Fluid Reserv...,WASHER FLUID RESERVOIR - FRONT (DORMAN 603-057),https://www.go-parts.com/ps-product-new.php?sk...,603-057,DORMAN,37.95
1,1,"[Mounting Bracket Included:No, Mounting Hardwa...","[Windshield Washer Fluid Reservoir, , , , Prod...",WASHER FLUID RESERVOIR - FRONT (DORMAN 603-120),https://www.go-parts.com/ps-product-new.php?sk...,603-120,DORMAN,65.69
2,2,"[Brand:Dorman, Condition:New]","[with Cap and Level Sensor, ]",WASHER FLUID RESERVOIR (DORMAN W0133-3043708),https://www.go-parts.com/ps-product-new.php?sk...,W0133-3043708,DORMAN,65.84
3,3,"[Cap Included:Yes, Washer Pump Included:Yes, S...","[with filler neck, , , Product Attributes:, , ...",WASHER FLUID RESERVOIR 4 CYL 1.8L (LKQ HO1288235),https://www.go-parts.com/ps-product-new.php?sk...,HO1288235,LKQ,77.76
4,4,"[Suggested purchase quantity:1, Brand:LKQ, Con...",[without headlight washer with cap ; without I...,WASHER FLUID RESERVOIR (LKQ GM1288245),https://www.go-parts.com/ps-product-new.php?sk...,GM1288245,LKQ,55.04
...,...,...,...,...,...,...,...,...
282,282,"[Brand:Dorman, Condition:New]",[],WASHER FLUID RESERVOIR 4 CYL 1.5L (DORMAN W013...,https://www.go-parts.com/ps-product-new.php?sk...,W0133-4080185,DORMAN,111.34
283,283,"[Mounting Bracket Included:No, Mounting Hardwa...","[Windshield Washer Fluid Reservoir, Standard R...",WASHER FLUID RESERVOIR - FRONT 4 CYL 2.3L (DOR...,https://www.go-parts.com/ps-product-new.php?sk...,603-047,DORMAN,163.23
284,284,"[Mounting Bracket Included:Yes, Mounting Hardw...","[Windshield Washer Fluid Reservoir, Excludes C...",WASHER FLUID RESERVOIR - FRONT 4 CYL 2.0L (DOR...,https://www.go-parts.com/ps-product-new.php?sk...,603-226,DORMAN,77.37
285,285,"[Mounting Bracket Included:No, Mounting Hardwa...","[Windshield Washer Fluid Reservoir, , , , Prod...",WASHER FLUID RESERVOIR - FRONT (DORMAN 603-129),https://www.go-parts.com/ps-product-new.php?sk...,603-129,DORMAN,113.03


# 5. Post Processing Data and Exporting

In [35]:
len(df['Links'].unique())

258

In [36]:
df=df.dropna()
df.shape

(253, 8)

In [37]:
df['Product']=search_text
df['Source']=Source

In [38]:
cols=["Sl.No",
"Name",
"Product",
"Current Price",
"Details",
"Attributes",
"Part Number",
"Links",
"Source"
] #
df=df[cols]

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 253 entries, 0 to 286
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sl.No          253 non-null    object 
 1   Name           253 non-null    object 
 2   Product        253 non-null    object 
 3   Current Price  253 non-null    float64
 4   Details        253 non-null    object 
 5   Attributes     253 non-null    object 
 6   Part Number    253 non-null    object 
 7   Links          253 non-null    object 
 8   Source         253 non-null    object 
dtypes: float64(1), object(8)
memory usage: 19.8+ KB


In [40]:
print(df.shape)
df.head()

(253, 9)


,Sl.No,Name,Product,Current Price,Details,Attributes,Part Number,Links,Source
0,0,WASHER FLUID RESERVOIR - FRONT (DORMAN 603-057),Washer Fluid Reservoir,37.95,[Dual Coolant / Windshield Washer Fluid Reserv...,"[Mounting Bracket Included:Yes, Mounting Hardw...",603-057,https://www.go-parts.com/ps-product-new.php?sk...,Go-Parts
1,1,WASHER FLUID RESERVOIR - FRONT (DORMAN 603-120),Washer Fluid Reservoir,65.69,"[Windshield Washer Fluid Reservoir, , , , Prod...","[Mounting Bracket Included:No, Mounting Hardwa...",603-120,https://www.go-parts.com/ps-product-new.php?sk...,Go-Parts
2,2,WASHER FLUID RESERVOIR (DORMAN W0133-3043708),Washer Fluid Reservoir,65.84,"[with Cap and Level Sensor, ]","[Brand:Dorman, Condition:New]",W0133-3043708,https://www.go-parts.com/ps-product-new.php?sk...,Go-Parts
3,3,WASHER FLUID RESERVOIR 4 CYL 1.8L (LKQ HO1288235),Washer Fluid Reservoir,77.76,"[with filler neck, , , Product Attributes:, , ...","[Cap Included:Yes, Washer Pump Included:Yes, S...",HO1288235,https://www.go-parts.com/ps-product-new.php?sk...,Go-Parts
4,4,WASHER FLUID RESERVOIR (LKQ GM1288245),Washer Fluid Reservoir,55.04,[without headlight washer with cap ; without I...,"[Suggested purchase quantity:1, Brand:LKQ, Con...",GM1288245,https://www.go-parts.com/ps-product-new.php?sk...,Go-Parts


In [41]:
dfAtt=df[['Part Number','Attributes']]
dfAtt=dfAtt.explode('Attributes')
dfAtt[['Attributes', 'Value']] = dfAtt['Attributes'].str.split(':',1, expand=True)


Exception ignored in: <function tqdm.__del__ at 0x0000024C0EDAFD30>
Traceback (most recent call last):
  File "c:\Users\Vikram.Vadhirajan\Anaconda3\lib\site-packages\tqdm\std.py", line 1162, in __del__
    self.close()
  File "c:\Users\Vikram.Vadhirajan\Anaconda3\lib\site-packages\tqdm\std.py", line 1291, in close
    if self.last_print_t < self.start_t + self.delay:
AttributeError: 'tqdm' object has no attribute 'last_print_t'
Exception ignored in: <function tqdm.__del__ at 0x0000024C0EDAFD30>
Traceback (most recent call last):
  File "c:\Users\Vikram.Vadhirajan\Anaconda3\lib\site-packages\tqdm\std.py", line 1162, in __del__
    self.close()
  File "c:\Users\Vikram.Vadhirajan\Anaconda3\lib\site-packages\tqdm\std.py", line 1291, in close
    if self.last_print_t < self.start_t + self.delay:
AttributeError: 'tqdm' object has no attribute 'last_print_t'
Exception ignored in: <function tqdm.__del__ at 0x0000024C0EDAFD30>
Traceback (most recent call last):
  File "c:\Users\Vikram.Vadhiraja

In [42]:
summary_df = pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count').reset_index()
summary_df =summary_df.sort_values(by='Value',ascending=False)
summary_df=summary_df.rename(columns ={'Value':'No Products contains this Attribute'})
summary_df


,Attributes,No Products contains this Attribute
6,Condition,253
1,Brand,253
25,Suggested purchase quantity,188
20,Package Contents,134
18,Mounting Hardware Included,133
23,Reservoir Cap Included,127
16,Mounting Bracket Included,127
7,Fluid Level Sensor,32
26,Washer Pump Included,27
5,Capacity,23


In [43]:
with pd.ExcelWriter(OFolder+'\\'+f'{Source}ProductDetails_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Raw')
    summary_df.to_excel(writer,index=False, sheet_name='Attribute_Summary')

# 99. Archived Codes

In [1]:
#Downloading images from amazon.
from selenium import webdriver
from selenium.webdriver.common.by import By
import openpyxl
import requests
from io import BytesIO
from openpyxl import Workbook
from openpyxl.drawing.image import Image

#--------------------------------------------------------------------------------------------------------------------------
il=[]
for i in range(length):
    driver.get(links[i])
    imagelink=driver.find_element(By.ID,"landingImage").get_attribute('src')
    il.append(imagelink)
    response = requests.get(imagelink)
    img_data = BytesIO(response.content)
    ASIN=links[i].split('dp/')[1]
    img_filename = f"{ASIN}.jpg"
    with open(img_filename, "wb") as img_file:
        img_file.write(img_data.getvalue())
    # # Create a DataFrame
    # df = pd.DataFrame({'Product Image URL': [imagelink]})
    # df.loc[count,'ASIN']=ASIN
    # # Save the DataFrame to an Excel file
    # excel_filename = "product_data.xlsx"
    # df.to_excel(excel_filename, index=False,)
    # wb = Workbook()
    # ws = wb.active
    # img = Image(img_filename)
    # ws.add_image(img, f'C{count+2}')
    # wb.save(excel_filename)
    balanceitem=length-i
    timeremaining(balanceitem,i)

NameError: name 'length' is not defined